In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import numpy as np
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

print("import 완료!")

import 완료!


In [ ]:
# ========== MNIST ==========
trainset_mnist = datasets.MNIST(root='./data', train=True, download=True,
                                transform=transforms.Compose([transforms.ToTensor()]))
testset_mnist = datasets.MNIST(root='./data', train=False, download=True,
                               transform=transforms.Compose([transforms.ToTensor()]))

MNIST_train = DataLoader(trainset_mnist, batch_size=32, shuffle=True, num_workers=2)
MNIST_test  = DataLoader(testset_mnist,  batch_size=32, shuffle=False, num_workers=2)

MNIST_train_images, MNIST_train_labels = [], []
for images, labels in MNIST_train:
    MNIST_train_images.append(images.view(images.shape[0], -1).numpy())
    MNIST_train_labels.append(labels.numpy())
MNIST_train_images = np.vstack(MNIST_train_images)
MNIST_train_labels = np.concatenate(MNIST_train_labels)

MNIST_test_images, MNIST_test_labels = [], []
for images, labels in MNIST_test:
    MNIST_test_images.append(images.view(images.shape[0], -1).numpy())
    MNIST_test_labels.append(labels.numpy())
MNIST_test_images = np.vstack(MNIST_test_images)
MNIST_test_labels = np.concatenate(MNIST_test_labels)

print("MNIST train:", MNIST_train_images.shape)
print("MNIST test :", MNIST_test_images.shape)

# ========== CIFAR-10 ==========
trainset_CIFAR = datasets.CIFAR10(root='./data', train=True, download=True,
                                  transform=transforms.Compose([transforms.ToTensor()]))
testset_CIFAR  = datasets.CIFAR10(root='./data', train=False, download=True,
                                  transform=transforms.Compose([transforms.ToTensor()]))

CIFAR_train = DataLoader(trainset_CIFAR, batch_size=32, shuffle=True, num_workers=2)
CIFAR_test  = DataLoader(testset_CIFAR,  batch_size=32, shuffle=False, num_workers=2)

CIFAR_train_images, CIFAR_train_labels = [], []
for images, labels in CIFAR_train:
    CIFAR_train_images.append(images.view(images.shape[0], -1).numpy())
    CIFAR_train_labels.append(labels.numpy())
CIFAR_train_images = np.vstack(CIFAR_train_images)
CIFAR_train_labels = np.concatenate(CIFAR_train_labels)

CIFAR_test_images, CIFAR_test_labels = [], []
for images, labels in CIFAR_test:
    CIFAR_test_images.append(images.view(images.shape[0], -1).numpy())
    CIFAR_test_labels.append(labels.numpy())
CIFAR_test_images = np.vstack(CIFAR_test_images)
CIFAR_test_labels = np.concatenate(CIFAR_test_labels)

print("CIFAR train:", CIFAR_train_images.shape)
print("CIFAR test :", CIFAR_test_images.shape)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.72MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.32MB/s]


MNIST train: (60000, 784)
MNIST test : (10000, 784)


100%|██████████| 170M/170M [00:13<00:00, 12.8MB/s]


CIFAR train: (50000, 3072)
CIFAR test : (10000, 3072)


In [ ]:
params_grid = {
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_leaf_nodes':    [5, 10, None]
}
depths = [3, 6, 9, 12]

def run_decision_tree(train_X, train_y, test_X, test_y, dataset_name):
    results = []
    for depth in depths:
        print(f"[{dataset_name}] depth={depth} 학습 중...")
        dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
        grid = GridSearchCV(dt, params_grid, cv=5, n_jobs=-1)
        grid.fit(train_X, train_y)

        best = grid.best_estimator_
        train_acc = accuracy_score(train_y, best.predict(train_X))
        test_acc  = accuracy_score(test_y,  best.predict(test_X))

        print(f"  → Best params: {grid.best_params_}")
        print(f"  → Train acc: {train_acc:.4f} | Test acc: {test_acc:.4f}\n")
        results.append({
            'Depth': depth,
            'Train Acc': round(train_acc, 4),
            'Test Acc': round(test_acc, 4)
        })

    df = pd.DataFrame(results)
    print(f"===== {dataset_name} Decision Tree 최종 결과 =====")
    print(df.to_string(index=False))
    return df

dt_mnist = run_decision_tree(MNIST_train_images, MNIST_train_labels,
                              MNIST_test_images,  MNIST_test_labels, "MNIST")

dt_cifar = run_decision_tree(CIFAR_train_images, CIFAR_train_labels,
                              CIFAR_test_images,  CIFAR_test_labels, "CIFAR-10")

[MNIST] depth=3 학습 중...
  → Best params: {'max_leaf_nodes': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
  → Train acc: 0.4915 | Test acc: 0.4953

[MNIST] depth=6 학습 중...
  → Best params: {'max_leaf_nodes': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
  → Train acc: 0.7382 | Test acc: 0.7415

[MNIST] depth=9 학습 중...
  → Best params: {'max_leaf_nodes': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
  → Train acc: 0.8665 | Test acc: 0.8501

[MNIST] depth=12 학습 중...
  → Best params: {'max_leaf_nodes': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
  → Train acc: 0.9491 | Test acc: 0.8778

===== MNIST Decision Tree 최종 결과 =====
 Depth  Train Acc  Test Acc
     3     0.4915    0.4953
     6     0.7382    0.7415
     9     0.8665    0.8501
    12     0.9491    0.8778
[CIFAR-10] depth=3 학습 중...
  → Best params: {'max_leaf_nodes': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
  → Train acc: 0.2376 | Test acc: 0.2394

[CIFAR-10] depth=6 학습 중...
  → Best params: {'m

In [ ]:
def run_svm(train_X, train_y, test_X, test_y, dataset_name):
    results = []
    for kernel in ['linear', 'rbf']:
        print(f"[{dataset_name}] SVM kernel={kernel} 학습 중...")
        clf = svm.SVC(kernel=kernel, random_state=42)
        clf.fit(train_X, train_y)

        train_acc = accuracy_score(train_y, clf.predict(train_X))
        test_acc  = accuracy_score(test_y,  clf.predict(test_X))

        print(f"  → Train acc: {train_acc:.4f} | Test acc: {test_acc:.4f}\n")
        results.append({
            'Kernel': kernel,
            'Train Acc': round(train_acc, 4),
            'Test Acc': round(test_acc, 4)
        })

    df = pd.DataFrame(results)
    print(f"===== {dataset_name} SVM 최종 결과 =====")
    print(df.to_string(index=False))
    return df

svm_mnist = run_svm(MNIST_train_images, MNIST_train_labels,
                    MNIST_test_images,  MNIST_test_labels,  "MNIST")

svm_cifar = run_svm(CIFAR_train_images, CIFAR_train_labels,
                    CIFAR_test_images,  CIFAR_test_labels,  "CIFAR-10")

[MNIST] SVM kernel=linear 학습 중...
  → Train acc: 0.9707 | Test acc: 0.9403

[MNIST] SVM kernel=rbf 학습 중...
  → Train acc: 0.9899 | Test acc: 0.9792

===== MNIST SVM 최종 결과 =====
Kernel  Train Acc  Test Acc
linear     0.9707    0.9403
   rbf     0.9899    0.9792
[CIFAR-10] SVM kernel=linear 학습 중...
  → Train acc: 0.5749 | Test acc: 0.3754

[CIFAR-10] SVM kernel=rbf 학습 중...
  → Train acc: 0.7028 | Test acc: 0.5436

===== CIFAR-10 SVM 최종 결과 =====
Kernel  Train Acc  Test Acc
linear     0.5749    0.3754
   rbf     0.7028    0.5436
